# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Classification. The real question behind "Ranking Signal Analysis" is "will this page decline?" that's a yes/no label from an observed outcome, not a priority score Iam inventing and not a grouping problem. It maps to row 2 of the framing table: Will this one decline / recover? → Classification.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label, derived as trend_direction == "down".This is an observed outcome (trend_direction comes from real trailing performance data), not a rule I invented. trend_direction and trend_pct are excluded from features,they're the source of the label, so using them as inputs would leak the answer into the model.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary: Precision@50 — of the top 50 content items my model ranks as most likely to be declining, how many actually are. This is the number that matters to the decision ("which pages does the editor open first"), not justo verall accuracy.Secondary: ROC-AUC vs. the base rate, to check the classifier is doing better than guessing before I even build a queue out of it.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [55]:
!rm -rf flyrank-ml-hasnain # Remove existing clone if any
!git clone https://github.com/HasnainRaza16/flyrank-ml-hasnain.git

import os
os.chdir('/content/flyrank-ml-hasnain')

Cloning into 'flyrank-ml-hasnain'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 150 (delta 56), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 1.88 MiB | 11.79 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [56]:
!find / -name "content_refresh_anonymized.csv" 2>/dev/null

/content/flyrank-ml-hasnain/flyrank-ml-hasnain/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-hasnain/data/raw/content_refresh_anonymized.csv


In [57]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-hasnain/flyrank-ml-hasnain/data/raw/content_refresh_anonymized.csv")

ranking_signal_cols = [
    "content_id", "client_id", "content_type",
    "avg_position", "ctr", "impressions_90d", "clicks_90d",
    "trend_direction", "trend_pct", "content_age_days",
    "days_since_last_update", "word_count",
]
lane_df = df[ranking_signal_cols].copy()

print(f"Rows: {len(lane_df)}")
lane_df.head()

Rows: 30000


,content_id,client_id,content_type,avg_position,ctr,impressions_90d,clicks_90d,trend_direction,trend_pct,content_age_days,days_since_last_update,word_count
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,0.76,3803,29,down,-41.4,187,20,3221.0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,0.05,15320,7,down,-57.7,445,25,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,0.09,12581,11,down,-60.9,141,20,3515.0
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,0.49,11751,58,stable,-13.8,463,22,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,0.13,19140,24,down,-34.7,263,14,2803.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A rule like "flag anything with avg_position worse than 20" ignores that decline shows up differently per content_type one type is missing ~100% of its keyword data, another ~28% of word_count, so a single global threshold either misses whole categories or false-flags them. Decline also comes from several signals moving together (avg_position, ctr, engagement_rate, scroll_rate, ai_traffic_pct), not one number crossing a line. The reference pipeline already shows this isn't hypothetical: the hand-rule baseline hits Precision@50 ≈ 0.24, the trained model hits ≈ 0.74 on the same data.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.